# 03 · The same analysis in SQL (DuckDB)

DuckDB runs SQL directly on the 320 MB CSV, with no database server. This notebook reproduces key results of notebook 02 using **CTEs, window functions, conditional aggregation and percentiles**, the SQL patterns data-analyst interviews ask about. Results must match the pandas version; that is the cross-check.

In [1]:
import sys
sys.path.append("../src")
import duckdb
from dnp_utils import latest_raw_file
con = duckdb.connect()
path = latest_raw_file().as_posix()
con.execute(f"""
CREATE VIEW projects AS
SELECT Bpin, EstadoProyecto AS state, Sector,
       CASE WHEN lower(TipoProyecto)='pgn' THEN 'PGN' WHEN lower(TipoProyecto)='sgr' THEN 'SGR' ELSE TipoProyecto END AS source,
       try_cast(split_part(Horizonte,'-',1) AS INT) AS start_year,
       try_cast(split_part(Horizonte,'-',2) AS INT) AS end_year,
       ValorTotalProyecto AS total_value, ValorVigenteProyecto AS current_value, ValorPagoProyecto AS paid
FROM read_csv('{path}', header=true, sample_size=-1)
""")
con.sql("SELECT count(*) AS projects, count(DISTINCT Bpin) AS unique_bpin FROM projects").df()

,projects,unique_bpin
0,530732,530732


## 1. Projects and value by source (`GROUP BY` + conditional aggregation)

In [2]:
con.sql("""
SELECT source,
       count(*)                                         AS projects,
       count(*) FILTER (WHERE total_value > 0)          AS valued,
       round(sum(total_value)/1e12, 1)                  AS total_trillion_cop,
       round(median(total_value) FILTER (WHERE total_value > 0)/1e6, 0) AS median_million_cop
FROM projects GROUP BY source ORDER BY projects DESC
""").df()

,source,projects,valued,total_trillion_cop,median_million_cop
0,Territorial,454026,167882,1142.9,210.0
1,SGR,65833,65427,416.6,1698.0
2,PGN,8216,5008,1495.8,10216.0
3,Obras por Impuestos,1884,1033,18.8,7585.0
4,AESGPRI,219,63,0.1,600.0
5,Esquemas Asociativos,197,110,6.4,1027.0
6,Localidades,173,149,0.1,110.0
7,EICE,128,98,16.1,12474.0
8,Proyectos por postulación,56,23,0.5,6309.0


## 2. Top sectors and their share of the total (window function)

In [3]:
con.sql("""
WITH s AS (
  SELECT Sector, count(*) AS projects, sum(total_value) AS total
  FROM projects WHERE total_value > 0 GROUP BY Sector
)
SELECT Sector, projects, round(total/1e12,1) AS total_trillion,
       round(100*total/sum(total) OVER (), 1) AS share_pct,
       rank() OVER (ORDER BY total DESC) AS rnk
FROM s ORDER BY rnk LIMIT 10
""").df()

,Sector,projects,total_trillion,share_pct,rnk
0,Transporte,31186,853.7,27.6,1
1,Educación,20933,514.5,16.6,2
2,Salud Y Protección Social,16200,357.1,11.5,3
3,Inclusión Social Y Reconciliación,22206,253.3,8.2,4
4,"Vivienda, Ciudad Y Territorio",24736,195.6,6.3,5
5,Minas Y Energía,7839,148.3,4.8,6
6,Agricultura Y Desarrollo Rural,15271,145.1,4.7,7
7,Gobierno Territorial,34489,111.2,3.6,8
8,Trabajo,2972,72.9,2.4,9
9,Hacienda,239,67.0,2.2,10


## 3. Concentration: share of value held by the top 1% of projects (`NTILE`)

In [4]:
con.sql("""
WITH ranked AS (
  SELECT total_value, ntile(100) OVER (ORDER BY total_value DESC) AS pct_bucket
  FROM projects WHERE total_value > 0
)
SELECT round(100*sum(total_value) FILTER (WHERE pct_bucket = 1)/sum(total_value), 1) AS top1pct_share
FROM ranked
""").df()

,top1pct_share
0,75.0


## 4. SGR: share finished by planned duration, with denominator and cohort control

In [5]:
con.sql("""
WITH sgr AS (
  SELECT state, end_year - start_year + 1 AS years
  FROM projects
  WHERE source='SGR' AND start_year <= 2021 AND end_year <= 2040
    AND state IN ('Terminado','En Ejecución','Formulación','Viable','Aprobado')
)
SELECT CASE WHEN years <= 3 THEN '1-3 yrs' WHEN years <= 10 THEN '4-10 yrs' ELSE '>10 yrs' END AS duration_band,
       count(*) AS projects,
       round(avg((state='Terminado')::INT), 3) AS finished_share
FROM sgr GROUP BY 1 ORDER BY min(years)
""").df()

,duration_band,projects,finished_share
0,1-3 yrs,24558,0.513
1,4-10 yrs,6875,0.869
2,>10 yrs,1675,0.907


Matches notebook 02 (0.513 / 0.869 / 0.907). Same answer from two tools is the check that the pandas logic is right.

In [6]:
con.sql("""
-- Year-over-year: projects starting each year by source (self-join free, using LAG)
WITH y AS (
  SELECT start_year, source, count(*) AS n
  FROM projects WHERE source IN ('Territorial','SGR','PGN') AND start_year BETWEEN 2012 AND 2026
  GROUP BY 1,2
)
SELECT start_year, source, n,
       n - lag(n) OVER (PARTITION BY source ORDER BY start_year) AS change_vs_prev_year
FROM y WHERE source='SGR' ORDER BY start_year
""").df()

,start_year,source,n,change_vs_prev_year
0,2012,SGR,2673,<NA>
1,2013,SGR,3942,1269
2,2014,SGR,2233,-1709
3,2015,SGR,3020,787
4,2016,SGR,1065,-1955
5,2017,SGR,2625,1560
6,2018,SGR,4410,1785
7,2019,SGR,5590,1180
8,2020,SGR,4503,-1087
9,2021,SGR,6425,1922
